<a href="https://colab.research.google.com/github/AmerFernandez/introduccion-ingenieria-sistemas/blob/main/EA1_Ingestio%CC%81n_de_Datos_desde_un_API_Amer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**<span style="color:#809bd8">Curso:</span>** **Introducción a Ingeniería de Sistemas**

**<span style="color:#809bd8">Tema:</span>** **Evidencia de Aprendizaje EA1**

**<span style="color:#809bd8">Estudiante:</span>** **Nadia Andrea Jaramillo Giraldo**

**<span style="color:#809bd8">Profesor:</span>** **Walter Hugo Arboleda Mazo**

**<span style="color:#809bd8">Universidad:</span>** **UNAC**

**<span style="color:#809bd8">Fecha:</span>** **21/02/2026**

In [ ]:
# pip install requests pandas openpyxl

In [ ]:
import requests # importacion de la libreria Requests para enviar peticion a la API https://jsonplaceholder.typicode.com/users
import sqlite3 # importacion de la interface para conexion y creacion de bases de datos SQLite
import pandas as pd # importacion de Pandas, libreria para el analisis de datos en Python

# Definicion de la URL de la API para envio de peticion y recepcion de datos usando protocolo HTTP
# desde la API https://jsonplaceholder.typicode.com/
# una API es una herramienta que se utilizan para facilitgar el trabajo a la hora de programar , conteniendo codigos que se pueden utilizar en diversas app si es necesario
# las API se clasifican segun su funcion , las mas usadas por internet son
# GET-obtener datos
# POST-enviar datos
# PUT-actualizar datos
# DELETE-borrarar datos
# en este se utiliza GET para solicitar datos
API_URL = "https://jsonplaceholder.typicode.com/users"


# Creacion de la funcion extraer_datos validandose codigo 200 (exito en la conexion) o codigo 404 (codigo de error en la conexion)
#los datos que nos arroja la API es de formato json
# json es un formato ligero utilizado para el intercambio de almacenamientos de datos estructurados
def extraer_datos(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Error al conectar con la API: {response.status_code}")


# Creacion de la base de datos en SQLite "datos_api.db"
# se realiza la conexion a dicha base de datos
# y se crea el "cursor" para "execute" el CREATE TABLE y el INSERT OR REPLACE INTO usuarios
# la funcion def nos sirve para crear una funcion
# curso es lo que nos permite ejecutar comando SQL , con el se pueden hacer cosas como crear tablas , insertar datos , consultar informacion
# el comando sqlite3.connect nos sirve para conectarnos a la base de datos SQL
# los datos quedan guardados en el archivo datos_api.db
# en caso de que el archivo no exista python lo crea automaticamente
def guardar_en_db(datos):
    conn = sqlite3.connect('datos_api.db')
    cursor = conn.cursor()

    # Creación de la tabla "usuarios" dentro de la base de datos "datos_api.db"
    # esta contiene los campos id, nombre, usuario y email, obtenidos desde la api
    # los datos guardados permiten consultar los usuarios mas adelante
    # la funcion IT NOT EXISTS sirve para que no se cree mas de una tabla en caso de que ya exista una , esto para evitar error cuando se ejecute muchas veces el programa
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            usuario TEXT,
            email TEXT
        )
    ''')

    # Inserción de datos en la tabla "usuarios"
    # encontramos todos los datos de los usuarios que tenemos en la tabla
    # con for item in datos podemos recorrer todos los usuarios dentro de la lista de datos c
    # la funcion INSERT OR REPLACE sirve para insertar un nuevo registra , si ya existe uno se reemplaza
    for item in datos:
        cursor.execute('''
            INSERT OR REPLACE INTO usuarios (id, nombre, usuario, email)
            VALUES (?, ?, ?, ?)
        ''', (item['id'], item['name'], item['username'], item['email']))

    # Aceptación mediante "Commit" el ingreso de datos en la tabla "usuarios"
    # confirma los cambios realizados en la base de datos
    # commit() permite guarda permanentemente los datos que se insertaron o se modificaron
    # si no se usa commit() no se guardaran los cambios en la base de datos
    conn.commit()

    # Cerrado de la conexion a la base de datos "datos_api.db"
     # es importante cerrar la conexion para liberar recursos del sistema
    # evita errorres si el programa se vuelve a conectar despues
    conn.close()

# 3. GENERAR EVIDENCIAS (Pandas y Auditoría)
# lee todos los datos que se encuentran en la tabla usuarios
# guarda los dotos en un archivos CSV
# sqlite3 abre la coneccion con la base de datos
# pd.read_sql_query ejecuta la consulta SQL y devuelve un dataframe de pandas
# lindex=false evita que pandas agrege una columna extra de indice
def generar_evidencias(datos_api):
    # --- Archivo Excel/CSV con Pandas ---
    conn = sqlite3.connect('datos_api.db')
    df_db = pd.read_sql_query("SELECT * FROM usuarios", conn)
    df_db.to_csv('muestra_usuarios.csv', index=False)
    conn.close()

    # la variable "registros_api" obtiene la cantidad de registros recibidos desde la
    # API con url "https://jsonplaceholder.typicode.com/users"
    # la variable "registros_db" obtiene la cantidad de registros almacenados en la tabla "usuarios"
    #  datos_api es la lista que tenemos y resivimos desde la api
    # len() cuenta cuantos usuarios tenemos en la api
    # sirve para comprobar que todos los datos se hayan guardado de manera correcta
    registros_api = len(datos_api)
    registros_db = len(df_db)

    # Creación del archivo "auditoría.txt" con el metodo "open" de Python
    # el archivo sirve para evidenciar una operacion cuando se inserta un dato
    # se utiliza encoding="utf-8" para soportar caracteres especiales
    with open('auditoria.txt', 'w', encoding='utf-8') as f:
        f.write("RESUMEN DE AUDITORÍA\n")
        f.write("====================\n")
        # escribir la cantidad de registros que hay en la API
        f.write(f"Registros extraídos de la API https://jsonplaceholder.typicode.com/: {registros_api}\n")
        # escribe la cantidad de registros almacenados en la base de datos
        f.write(f"Registros guardados en la DB de SQLite3: {registros_db}\n")
        # comparar la cantidad de registros para verificar que todo se haya guardado correctamente

        if registros_api == registros_db:
            f.write("\nESTADO: Operacion Exitosa.\n")
            # si coinciden , la operacion fue exitosa
        else:
            f.write("\nESTADO: Operacion Fallida.\n")
             # si no coinciden , hubo algun error al insertar un dato

# EJECUCIÓN DEL PROCESO
try:
    print("Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...")
    datos = extraer_datos(API_URL)

    print("Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...")
    guardar_en_db(datos)

    print("Generando archivos de evidencia .CSV y auditoría.txt...")
    generar_evidencias(datos)

    print("¡Proceso completado con éxito!")
except Exception as e:
    print(f"Ocurrió un error: {e}")

Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...
Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...
Generando archivos de evidencia .CSV y auditoría.txt...
¡Proceso completado con éxito!



**Referencias**

https://realpython.com/python-requests/#make-a-get-request

https://pandas.pydata.org/

https://pypi.org/project/openpyxl/

https://docs.python.org/3/library/sqlite3.html

https://sqliteviewer.app/

https://sibabalwesinyaniso.medium.com/what-is-a-cursor-understanding-how-sqlite-handles-queries-in-python-8f8c88546820
https://www.pythonlore.com/understanding-sqlite3-cursor-object-for-database-operations/
https://miro.medium.com/v2/resize:fit:720/format:webp/0*rePKZ6jMZbZ_6S_3.gif
